In [ ]:
import os
import PIL as pil
from tqdm import tqdm
import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as F
from torchsummary import summary
from torchvision.utils import make_grid
import cv2
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
IMG_SIZE = 256
IMG_DIR = "data/"
BATCH_SIZE = 32
NUM_WORKERS = 8

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Assuming that we are on a CUDA machine, this should print a CUDA device:

print(device)

In [ ]:
def get_folder_structure(base="data"):
    """
    Returns the folder structure of the data folder
    """
    return [x[0] for x in os.walk(base)]


get_folder_structure()

In [ ]:
class BenGrahamPreprocessing:
    def __init__(self, img_size):
        self.img_size = img_size

    def __call__(self, image: pil.Image.Image) -> pil.Image.Image:
        image_np = np.array(image)

        # Aplicar el desenfoque gaussiano de OpenCV
        blurred_image = cv2.GaussianBlur(image_np, (0, 0), self.img_size / 10)

        # Combinar la imagen original y la desenfocada
        combined_image = cv2.addWeighted(image_np, 4, blurred_image, -4, 128)

        # Convertir a escala de grises
        return F.to_pil_image(combined_image)

In [ ]:
transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        #  transforms.Grayscale(),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(30),
        BenGrahamPreprocessing(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean = [0.485, 0.456, 0.406],std = [0.229, 0.224, 0.225]),
    ]
)

In [ ]:
# Load the training data from the data folder
trainset = torchvision.datasets.ImageFolder(root="data/train/", transform=transform)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS
)
testset = torchvision.datasets.ImageFolder(root="data/test/", transform=transform)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS
)

classes = ("0-noDR", "1-mild", "2-moderate", "3-severe", "4-proliferativeDR")

In [ ]:
print("Follwing classes are there : \n", trainset.classes)

In [ ]:
def display_img(img, label):
    print(f"Label : {trainset.classes[label]}")
    plt.imshow(img.permute(1, 2, 0), cmap="gray")


# display the first image in the dataset
display_img(*trainset[0])

In [ ]:
num_classes = len(trainset.classes)
# Cargar un modelo preentrenado
model = torchvision.models.resnet152(weights=torchvision.models.ResNet152_Weights.IMAGENET1K_V2)

# Freeze all layers except the last ones
for param in model.parameters():
    param.requires_grad = False

# Modify the final layer for our classification task
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)

# Move model to device
model = model.to(device)

In [ ]:
summary(model, (3, IMG_SIZE, IMG_SIZE), device=device.type)

In [ ]:
criterion = torch.nn.CrossEntropyLoss()  # Use CrossEntropy for multi-class classification
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)  # Only optimize parameters of final layer
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20)

In [ ]:
torch.set_num_threads(os.cpu_count()-2)  # Usa todos los cores

In [ ]:
def train_model(model, criterion, optimizer, num_epochs=10):
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in tqdm(trainloader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
        epoch_loss = running_loss / len(trainloader.dataset)
        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}')
        # Validation loop
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in tqdm(testloader):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_accuracy = correct / total
        print(f'Validation Accuracy: {val_accuracy:.4f}')

In [ ]:
# def train_model(model, criterion, optimizer, num_epochs=10):
#     """
#     Entrena el modelo.

#     Args:
#         model: El modelo a entrenar.
#         criterion: La función de pérdida.
#         optimizer: El optimizador.
#         num_epochs: El número de épocas de entrenamiento.

#     Returns:
#         El modelo entrenado.
#     """
#     for epoch in range(num_epochs):
#         print(f"Epoch {epoch}/{num_epochs - 1}")
#         print("-" * 10)

#         # Iterar sobre las fases de entrenamiento y validación
#         for phase in ["train"]:  # , 'val']:
#             if phase == "train":
#                 model.train()  # Poner el modelo en modo de entrenamiento
#             else:
#                 model.eval()  # Poner el modelo en modo de evaluación

#             running_loss = 0.0
#             running_corrects = 0

#             # Iterar sobre los datos en batches
#             for inputs, labels in tqdm(trainloader):  # dataloaders[phase]:
#                 inputs = inputs.to(device)  # Mover los datos al dispositivo
#                 labels = labels.to(device)

#                 # Zero the parameter gradients
#                 optimizer.zero_grad()

#                 # Habilitar el cálculo de gradientes solo en la fase de entrenamiento
#                 with torch.set_grad_enabled(phase == "train"):
#                     # Forward pass: calcular las predicciones
#                     outputs = model(inputs)
#                     _, preds = torch.max(outputs, 1)  # Obtener la clase predicha
#                     loss = criterion(outputs, labels)  # Calcular la pérdida

#                     # Backward pass + optimizar solo en la fase de entrenamiento
#                     if phase == "train":
#                         loss.backward()  # Calcular los gradientes
#                         optimizer.step()  # Actualizar los pesos del modelo

#                 # Calcular estadísticas
#                 running_loss += loss.item() * inputs.size(0)
#                 running_corrects += torch.sum(preds == labels.data)

#             # Calcular la pérdida y la precisión para la época
#             epoch_loss = running_loss / len(trainloader.dataset)  # dataset_sizes[phase]
#             epoch_acc = running_corrects.double() / len(trainloader.dataset)  # dataset_sizes[phase]

#             print(f"{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

#         print()

#     print("Entrenamiento completado")
#     return model

In [ ]:
train_model(model, criterion, optimizer, num_epochs=10)

In [ ]:
model.eval()  # Establece modo de evaluación
correct = 0
total = 0

with torch.no_grad():  # Desactiva el cálculo de gradientes para evaluación
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Precisión en el conjunto de prueba: {100 * correct / total:.2f}%")